# <u> The Local "_Stars Appearing_": one constellation </u>
### Sonification and Visuals for Planetaria

The _"Stars Appearing"_ piece from ["Audio Universe: Tour of the Solar System"](https://www.audiouniverse.org/education-and-communication/educational-shows/tour-of-the-solar-system), for **a single constellation**, as seen from **any site, on any night**. You can listen to a constellation appear!

This is the companion to the [all-sky notebook, `StarsAppearingColab.ipynb`](https://githubtocolab.com/Audio-Universe/sonified-night-sky/blob/main/StarsAppearingColab.ipynb), and works the same way - only the stars in the chosen constellations 'asterism' are represented, for introducing one constellation during a show.

 The brightest of its stars is heard first, then dimmer and dimmer ones, and the piece falls quiet once the figure is complete.

The figures are the ones _Stellarium_ draws - the same set the [Sonification Suite](https://www.audiouniverse.org/sonification-suite) uses.

You can produce the visuals in an equirectangular and/or full-dome format. The sounds can be exported in mono (with no spatial information), stereo, 5.1 or 7.1.

A constellation is only a handful of stars, so this is quicker than the all-sky piece at the same settings - but a full resolution render still takes much longer than a preview. **We recommend a preview size and stereo output while you are choosing.**

The first time you run the code within a given Colab session, we download the catalogues and a sky map (~100 MB), so will take longer.

**Note:** a constellation has to be above the horizon at the time you choose. If it is not, this will stop and tell you which ones are, so you can pick another - or another night - and run again.

**To get started:** Set up your sky via the form below, then go to `Runtime` > `Run all`.


In [ ]:
#@title Setup: fetch what this needs - run this first { display-mode: "form" }
import sys

# Getting the right assets from github
STRAUSS_REF = ("git+https://github.com/james-trayford/strauss.git"
               "@v1p5")
HELPER_URL = ("https://raw.githubusercontent.com/Audio-Universe/"
              "sonified-night-sky/main/StarsAppearingLocal.py")

if "google.colab" in sys.modules:
    from pathlib import Path

    try:
        import skyfield, strauss, timezonefinder
    except ImportError:
        !pip install --quiet "strauss @ {STRAUSS_REF}" skyfield==1.53 timezonefinder

    !wget --quiet -O StarsAppearingLocal.py "{HELPER_URL}"

In [1]:
#@title Settings: where, when, which constellation, and what to render { display-mode: "form" }
#@markdown ### Observer Location and Time
latitude = 53.1143737  #@param {type:"number"}
longitude = -1.2219389  #@param {type:"number"}
#@markdown Latitude is +ve north, longitude +ve east - 1.22&deg; west is `-1.22`.
datetime = "2026-09-19 19:00:00"  #@param {type:"string"}
#@markdown `datetime` is the local time in YYYY-MM-DD HH:mm:ss - The
#@markdown timezone is worked out from the coordinates, with summer time
#@markdown applied or not according to the date.
#@markdown

#@markdown `facing` is the centre of the panorama: the assumed
#@markdown forward-looking direction for an observer.
facing = "S"  #@param ["N","NNE","NE","ENE","E","ESE","SE","SSE","S","SSW","SW","WSW","W","WNW","NW","NNW"]

#@markdown ### The constellation
#@markdown Only the stars this figure is drawn from are sonified. It has to
#@markdown be above the horizon at the time set above - if none of it is,
#@markdown this stops and lists the ones that are.
constellation = "Cygnus"  #@param ["Andromeda", "Antlia", "Apus", "Aquarius", "Aquila", "Ara", "Aries", "Auriga", "Bootes", "Caelum", "Camelopardalis", "Cancer", "Canes Venatici", "Canis Major", "Canis Minor", "Capricornus", "Carina", "Cassiopeia", "Centaurus", "Cepheus", "Cetus", "Chamaeleon", "Circinus", "Columba", "Coma Berenices", "Corona Australis", "Corona Borealis", "Corvus", "Crater", "Crux", "Cygnus", "Delphinus", "Dorado", "Draco", "Equuleus", "Eridanus", "Fornax", "Gemini", "Grus", "Hercules", "Horologium", "Hydra", "Hydrus", "Indus", "Lacerta", "Leo", "Leo Minor", "Lepus", "Libra", "Lupus", "Lynx", "Lyra", "Mensa", "Microscopium", "Monoceros", "Musca", "Norma", "Octans", "Ophiuchus", "Orion", "Pavo", "Pegasus", "Perseus", "Phoenix", "Pictor", "Pisces", "Piscis Austrinus", "Puppis", "Pyxis", "Reticulum", "Sagitta", "Sagittarius", "Scorpius", "Sculptor", "Scutum", "Serpens", "Sextans", "Taurus", "Telescopium", "Triangulum", "Triangulum Australe", "Tucana", "Ursa Major", "Ursa Minor", "Vela", "Virgo", "Volans", "Vulpecula"]
#@markdown `mag_limit` does two jobs here. It sets how faint a star can be
#@markdown and still be part of the figure - a few figures reach magnitude 6,
#@markdown so lowering this drops their faintest stars - and it sets the
#@markdown pacing, since a star sounds at the moment it would in an all-sky
#@markdown piece drawn to the same limit.
mag_limit = 6  #@param {type:"slider", min:1, max:7, step:0.5}

#@markdown ### Sonification Properties
#@markdown A constellation needs far less time than a whole sky - its stars
#@markdown are all bright, so they sound in the early part of the piece.
duration = 12  #@param {type:"number"}
system = "stereo"  #@param ["mono","stereo","5.1","7.1"]
#@markdown `sound` is the sound and harmony style used.
sound = "Night Harp"  #@param ["Night Harp", "Glockenspiel"]

#@markdown ### The picture
#@markdown `output` is the shape of the video. `panorama` is rectangular - the
#@markdown equirectangular 360 x 180 degree view. `dome` is square - the
#@markdown fisheye planetarium master. `both` writes one of each.
output = "dome"  #@param ["panorama","dome","both"]
#@markdown `size` is the pixel resolution, for panorama (dome):
#@markdown `fast_preview` 512x256 (256), `preview` 1024x512 (512),
#@markdown `high` 4096x2048 (2048), `full` 8192x4096 (4096). The largest two are
#@markdown the native sizes of the star maps. Higher resolutions are slower.
size = "preview"  #@param ["fast_preview","preview","high","full"]
fps = 15  #@param {type:"integer"}
#@markdown `sky_exposure` is how bright the generated sky comes out, and
#@markdown `horizon` blacks out everything below the horizon.
sky_exposure = 0.75  #@param {type:"number"}
horizon = False  #@param {type:"boolean"}

from StarsAppearingLocal import (Config, make_sequence, show_videos,
                                 download_outputs, describe_when)

cfg = Config(latitude=latitude, longitude=longitude, date_time=datetime,
             facing=facing, mag_limit=mag_limit, constellation=constellation,
             duration=duration, system=system, size=size, fps=fps,
             output=output, sky_exposure=sky_exposure, horizon=horizon)

# the timezone was worked out rather than typed, so say which one it was
print(describe_when(cfg))


2026-09-19 19:00 Europe/London - BST (UTC+01:00)


### <u> Making it </u>

Generate the sky, the sonification and the animation - can take a while...

Progress bars are used to show different elements. If the constellation has
not risen, this is where it stops and tells you which ones have.


In [ ]:
result = make_sequence(cfg, sound=sound)

### <u> Show the Sequence </u>

In [ ]:
show_videos(result.videos)

### <u> Keeping it </u>

These files live on a temporary machine that _Colab_ throws away when your session ends, so download anything you want to keep.

Running the cell below once sequences have been generated will give you some handy buttons to download the full movie(s) and the audio file 

In [ ]:
download_outputs(result)